In [1]:
# Cài đặt thư viện xử lý mã băm hình ảnh
!pip install imagehash tqdm -q

import os
import shutil
from PIL import Image, ImageOps
import imagehash
from tqdm import tqdm

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THÔNG SỐ
# ==========================================
# TRỎ VÀO THƯ MỤC CHỨA 3 FOLDER TRAIN, VALID, TEST CỦA BẠN (Cần quyền Ghi/Write)
DATASET_DIR = '02042026.v2i.yolov8'

# Ngưỡng sai số (Khoảng cách Hamming).
# <= 4 là mức cực kỳ an toàn để tóm gọn ảnh đổi sáng/nhiễu mà không bắt nhầm lá khác.
THRESHOLD = 4

# ==========================================
# 2. HÀM TÍNH TOÁN "MÃ GEN" (HASH)
# ==========================================
def get_hash_signatures(img_path):
    try:
        img = Image.open(img_path).convert("RGB")
        # Tính hash cho 4 tư thế: Gốc, Lật ngang, Lật dọc, Lật chéo
        h_orig = imagehash.phash(img)
        h_hf = imagehash.phash(ImageOps.mirror(img))
        h_vf = imagehash.phash(ImageOps.flip(img))
        h_hv = imagehash.phash(ImageOps.flip(ImageOps.mirror(img)))
        return [h_orig, h_hf, h_vf, h_hv]
    except:
        return []

# ==========================================
# 3. THU THẬP DỮ LIỆU & TÍNH HASH
# ==========================================
print("🔍 BƯỚC 1: Đang quét toàn bộ ảnh và tính toán mã băm (Có thể mất 2-3 phút)...")
all_images = [] # Lưu: (path_img, path_lbl, split, hashes)

for split in ['train', 'valid', 'test', 'val']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')

    if not os.path.exists(img_dir): continue

    for img_name in tqdm(os.listdir(img_dir), desc=f"Quét tập {split}"):
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')): continue

        img_p = os.path.join(img_dir, img_name)
        lbl_p = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + '.txt')

        if os.path.exists(lbl_p):
            hashes = get_hash_signatures(img_p)
            if hashes:
                all_images.append({
                    'img_path': img_p,
                    'lbl_path': lbl_p,
                    'split': split if split != 'val' else 'valid',
                    'hashes': hashes
                })

n = len(all_images)
print(f"✅ Đã thu thập và tính Hash xong cho {n} bức ảnh.")

# ==========================================
# 4. TÌM KIẾM "GIA ĐÌNH" BẰNG THUẬT TOÁN UNION-FIND
# ==========================================
print("🕵️‍♂️ BƯỚC 2: Đang đối chiếu để tóm gọn các ảnh rò rỉ (Data Leakage)...")
parent = {i: i for i in range(n)}

def find(i):
    if parent[i] == i: return i
    parent[i] = find(parent[i])
    return parent[i]

def union(i, j):
    root_i = find(i)
    root_j = find(j)
    if root_i != root_j:
        parent[root_i] = root_j

# So sánh chéo tất cả các ảnh
for i in tqdm(range(n), desc="Đang phân tích độ tương đồng"):
    hashes_i = all_images[i]['hashes']
    for j in range(i + 1, n):
        # Nếu đã chung gốc thì bỏ qua
        if find(i) == find(j): continue

        hashes_j = all_images[j]['hashes']
        # Kiểm tra xem có bất kỳ tư thế nào của ảnh J giống ảnh I không
        is_match = False
        for hi in hashes_i:
            for hj in hashes_j:
                if hi - hj <= THRESHOLD: # Trừ hash sẽ ra khoảng cách Hamming
                    is_match = True
                    break
            if is_match: break

        if is_match:
            union(i, j)

# Gom nhóm theo Root
groups = {}
for i in range(n):
    root = find(i)
    if root not in groups:
        groups[root] = []
    groups[root].append(all_images[i])

# ==========================================
# 5. RA QUYẾT ĐỊNH & DI CHUYỂN FILE
# ==========================================
print("🚚 BƯỚC 3: Đang dọn dẹp và di chuyển các file sai vị trí...")
moved_count = 0

train_img_dir = os.path.join(DATASET_DIR, 'train', 'images')
train_lbl_dir = os.path.join(DATASET_DIR, 'train', 'labels')
os.makedirs(train_img_dir, exist_ok=True)
os.makedirs(train_lbl_dir, exist_ok=True)

for root_id, group in groups.items():
    # Kiểm tra điều kiện phải chuyển vào Train:
    # 1. Gia đình có nhiều hơn 1 người (ảnh bị augment/nhân bản)
    # 2. Hoặc có bất kỳ thành viên nào đang nằm ở tập Train
    is_leaked = len(group) > 1 or any(item['split'] == 'train' for item in group)

    if is_leaked:
        for item in group:
            if item['split'] != 'train':
                # Chuyển file vật lý sang tập Train
                img_dest = os.path.join(train_img_dir, os.path.basename(item['img_path']))
                lbl_dest = os.path.join(train_lbl_dir, os.path.basename(item['lbl_path']))

                shutil.move(item['img_path'], img_dest)
                shutil.move(item['lbl_path'], lbl_dest)

                # Cập nhật lại trạng thái để tránh lỗi
                item['split'] = 'train'
                item['img_path'] = img_dest
                item['lbl_path'] = lbl_dest
                moved_count += 1

print(f"🎉 HOÀN TẤT CHIẾN DỊCH THANH LỌC!")
print(f"👉 Đã phát hiện và bế thành công {moved_count} file rò rỉ từ Valid/Test tống về tập Train.")

🔍 BƯỚC 1: Đang quét toàn bộ ảnh và tính toán mã băm (Có thể mất 2-3 phút)...


Quét tập test: 100%|██████████| 963/963 [00:33<00:00, 28.71it/s] 


✅ Đã thu thập và tính Hash xong cho 9642 bức ảnh.
🕵️‍♂️ BƯỚC 2: Đang đối chiếu để tóm gọn các ảnh rò rỉ (Data Leakage)...


Đang phân tích độ tương đồng: 100%|██████████| 9642/9642 [32:43<00:00,  4.91it/s]  


🚚 BƯỚC 3: Đang dọn dẹp và di chuyển các file sai vị trí...
🎉 HOÀN TẤT CHIẾN DỊCH THANH LỌC!
👉 Đã phát hiện và bế thành công 1127 file rò rỉ từ Valid/Test tống về tập Train.
